<a href="https://colab.research.google.com/github/raul94IA/LLAMA_META/blob/main/Copia_de_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 12.9 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
# Nota Personal: el comando solicita una respuesta al modelo mandando el mensaje prompt con el rol de usuario
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco Duro / SSD)**  

| Característica | RAM | Almacenamiento (HDD/SSD) |
|----------------|-----|--------------------------|
| **Tipo** | Memoria volátil | Memoria no volátil |
| **Propósito principal** | Guardar datos e instrucciones que la CPU necesita **en ese instante** | Conservar datos de forma permanente, incluso cuando la computadora está apagada |
| **Velocidad** | Muy rápida (nanosegundos) | Más lenta (microsegundos a milisegundos) |
| **Capacidad típica** | 4 GB – 32 GB (en PCs de consumo) | 256 GB – 4 TB (o más, especialmente en servidores) |
| **Costo por GB** | Mayor | Menor |
| **Persistencia** | Se borra al apagar la máquina | Se mantiene intacto |
| **Acceso** | Directo y aleatorio | También aleatorio, pero con mayor latencia y, en discos mecánicos, con movimientos físicos |
| **Ejemplo de uso** | Ejecutar un programa, cargar una página web en el navegador, trabajar en una hoja de cálculo | Guardar archivos de d

In [ ]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa
#Nota personal: Le da formato a la informacion para que se pueda apreciar de forma mas estructurada

{
  "id": "chatcmpl-7ac9b48f-a545-4701-a933-2a04d19e587c",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "**RAM (Memoria de Acceso Aleatorio) vs. Almacenamiento (Disco Duro / SSD)**  \n\n| Característica | RAM | Almacenamiento (HDD/SSD) |\n|----------------|-----|--------------------------|\n| **Tipo** | Memoria volátil | Memoria no volátil |\n| **Propósito principal** | Guardar datos e instrucciones que la CPU necesita **en ese instante** | Conservar datos de forma permanente, incluso cuando la computadora está apagada |\n| **Velocidad** | Muy rápida (nanosegundos) | Más lenta (microsegundos a milisegundos) |\n| **Capacidad típica** | 4 GB – 32 GB (en PCs de consumo) | 256 GB – 4 TB (o más, especialmente en servidores) |\n| **Costo por GB** | Mayor | Menor |\n| **Persistencia** | Se borra al apagar la máquina | Se mantiene intacto |\n| **Acceso** | Directo y aleatorio | También aleatorio, pero con ma

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
# Nota Personal: Los toques totales es el costo total de la llamada
print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)

Tokens del prompt: 86
Tokens de la respuesta: 790
Tokens totales: 876


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
#Nota personal: secuencia: guarda el tiempo actual en "inicio" - manda mensaje al modelo - Obtiene la
#diferencia entre el tiempo actual y el tiempo del inico para obtener la duracion de la llamada al modelo
import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 1.21 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Modelo ligero: 1.21 s — 876 tokens
Modelo grande: 1.88 s — 737 tokens

Respuesta del modelo grande:
 **RAM (memoria de acceso aleatorio) vs. almacenamiento (disco duro/SSD)**  

| Característica | RAM | Almacenamiento |
|----------------|-----|----------------|
| **Función principal** | Guarda temporalmente los datos y programas que la CPU está usando en ese momento. | Guarda permanentemente el sistema operativo, aplicaciones, archivos, fotos, videos, etc. |
| **Volatilidad** | **Volátil**: al apagar o reiniciar el equipo, su contenido se pierde. | **No volátil**: la información permanece aunque el equipo esté apagado. |
| **Velocidad** | Mucho más rápida (nanosegundos). Permite accesos casi instantáneos a cualquier posición de la memoria. | Más lenta (micro‑ a milisegundos). Incluso los SSD son cientos de veces más lentos que la RAM. |
| **Tipo de tecnología** | Chips de DRAM (o SRAM en cachés). | Discos duros magnéticos (HDD) o unidades de estado sólido (SSD, NVMe). |
| **Capacidad t

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [ ]:
# Definir la lista de preguntas
preguntas = []
preguntas.append("¿Que es un PLC en automatizacion?")
preguntas.append("¿Que es un servomotor?")
preguntas.append("¿Que es un HMI en automatizacion?")
primera_pregunta = preguntas[0]
segunda_pregunta = preguntas[1]
tercera_pregunta = preguntas[2]
print(f"Pregunta 1: {primera_pregunta}")
print(f"Pregunta 2: {segunda_pregunta}")
print(f"Pregunta 3: {tercera_pregunta}")

Pregunta 1: ¿Que es un PLC en automatizacion?
Pregunta 2: ¿Que es un servomotor?
Pregunta 3: ¿Que es un HMI en automatizacion?


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
inicio = time.time()
response_1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": primera_pregunta}]
)

duracion_1 = time.time() - inicio

resultado_1 = {
    "pregunta": primera_pregunta,
    "respuesta": response_1.choices[0].message.content,
    "duracion": duracion_1,
    "tokens": response_1.usage.total_tokens
}

print(resultado_1["respuesta"])

### ¿Qué es un PLC (Controlador Lógico Programable) en automatización?

Un **PLC** (Programable Logic Controller, o *Controlador Lógico Programable* en español) es una computadora industrial diseñada para controlar procesos y máquinas en entornos de producción, procesos químicos, plantas de energía, sistemas de transporte, etc. Su principal función es **leer** señales de entrada (sensores, interruptores, botones, etc.), **ejecutar** una lógica preprogramada y **generar** señales de salida (relés, actuadores, válvulas, motores, etc.) para controlar el proceso.

---

## 1. Componentes básicos de un PLC

| Componente | Función principal |
|------------|-------------------|
| **CPU (Unidad Central de Procesamiento)** | Ejecuta el programa, gestiona la memoria y controla el flujo de datos. |
| **Fuente de alimentación** | Provee energía eléctrica estable (usualmente 24 V DC). |
| **Modulos de entrada** | Conectan sensores y dispositivos que envían datos al PLC (digitales o analógicos). |
| 

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
inicio = time.time()
response_2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": segunda_pregunta}]
)

duracion_2 = time.time() - inicio

resultado_2 = {
    "pregunta": segunda_pregunta,
    "respuesta": response_2.choices[0].message.content,
    "duracion": duracion_2,
    "tokens": response_2.usage.total_tokens
}

print(resultado_2["respuesta"])

Un **servomotor** es un tipo de motor eléctrico diseñado para ofrecer un control preciso de la posición, velocidad y aceleración. A diferencia de un motor típico que gira continuamente, un servomotor trabaja en ciclos de movimiento controlados, de modo que la salida (el eje del motor) llega a la posición exacta que se indica mediante una señal de control.

### Componentes básicos

| Parte | Función |
|-------|---------|
| **Motor (motor de corriente continua o de paso)** | Genera el movimiento rotacional. |
| **Potenciómetro o sensor de posición** | Detecta la posición real del eje y la compara con la posición deseada. |
| **Circuito de control (puente de H, microcontrolador, etc.)** | Procesa la señal de entrada y la señal de retroalimentación, ajustando la potencia que se aplica al motor. |
| **Alimentación eléctrica** | Proporciona la energía necesaria al motor. |

### ¿Cómo funciona?

1. **Señal de entrada** – Se envía una señal de control (por ejemplo, un pulso PWM de 1–2 ms dentr

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
inicio = time.time()
response_3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": tercera_pregunta}]
)

duracion_3 = time.time() - inicio

resultado_3 = {
    "pregunta": tercera_pregunta,
    "respuesta": response_3.choices[0].message.content,
    "duracion": duracion_3,
    "tokens": response_3.usage.total_tokens
}

print(resultado_3["respuesta"])

**HMI = Human‑Machine Interface**

En la automatización industrial, el **HMI (Human‑Machine Interface)** es el puente de comunicación entre el operador humano y la maquinaria o sistema de control. Su función principal es permitir que el ser humano vea lo que está sucediendo en el proceso y, al mismo tiempo, pueda enviar comandos o ajustar parámetros.

---

## 1. ¿Qué hace exactamente un HMI?

| Función | Detalle |
|---------|---------|
| **Visualización** | Pantallas gráficas (LCD, LED, táctiles) que muestran el estado real de la planta: valores de temperatura, presión, velocidad, alarmas, gráficas históricas, etc. |
| **Control** | Botones, diales, sliders, menús que permiten al operador activar/desactivar equipos, cambiar set‑points, ajustar parámetros de proceso. |
| **Alarma y diagnóstico** | Notificaciones visuales y sonoras cuando algo sale del rango normal. A menudo incluyen el historial de eventos y la posibilidad de registrar incidentes. |
| **Gestión de datos** | Registro de 

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
print(f"Pregunta {resultados[0]["pregunta"]} — tokens {resultados[0]["tokens"]} - duracion {resultados[0]["duracion"]:.2f} s")
print(f"Pregunta {resultados[1]["pregunta"]} — tokens {resultados[1]["tokens"]} - duracion {resultados[1]["duracion"]:.2f} s")
print(f"Pregunta {resultados[2]["pregunta"]} — tokens {resultados[2]["tokens"]} - duracion {resultados[2]["duracion"]:.2f} s")
#Las 3 respuestas me parecen lo bastante completas, compactas y precisas.

Pregunta ¿Que es un PLC en automatizacion? — tokens 1448 - duracion 1.84 s
Pregunta ¿Que es un servomotor? — tokens 895 - duracion 1.26 s
Pregunta ¿Que es un HMI en automatizacion? — tokens 1524 - duracion 1.92 s
